# Air Quality PM2.5 Forecasting Pipeline Demo

This notebook runs the end-to-end pipeline for the PM2.5 project.
Run cells top to bottom for a clean demo.

**Flow:** Ingest -> ETL -> EDA -> Features -> Train -> Predict


## 0) Quick Setup Check
Make sure you are using the correct Python kernel (venv).


In [19]:
import sys
print('Python:', sys.version)
print('Executable:', sys.executable)

Python: 3.10.13 (main, Jan 14 2026, 09:13:55) [GCC 13.3.0]
Executable: /home/parallels/aqi-bigdata/venv/bin/python


In [7]:
import os
PROJECT = os.path.abspath(os.path.join(os.getcwd(), '..'))
os.chdir(PROJECT)
print('CWD =', os.getcwd())


CWD = /home/parallels/aqi-bigdata


## 1) Ingestion (OpenAQ API)
This step fetches new raw JSON data and stores it in `data/raw/`.


In [8]:
!python src/ingest_openaq.py

🚀 OpenAQ Multi-Sensor Ingest
Sensors: ['35', '12234787', '40', '218', '219', '220', '5079217', '5077838', '1044', '35374', '35661', '4099', '67', '50', '61', '4202', '4154', '4118', '4610', '115', '4221', '4235', '4212', '4390', '4541', '4344', '4215', '206', '261', '1304692', '28890', '246', '249', '254', '257', '232', '12626392', '12915957', '235', '1927060', '228', '268', '2071327', '2071333', '2031', '2071339', '5077638', '453']
Date range: 2023-01-01T00:00:00Z to 2025-01-01T00:00:00Z
Interval: hourly (to maximize rows)

📡 Fetching sensor 35 (hourly)...
   URL: https://api.openaq.org/v3/sensors/35/measurements
   Params: {'limit': 100, 'page': 1, 'interval': 'hourly', 'datetime_from': '2023-01-01T00:00:00Z', 'datetime_to': '2025-01-01T00:00:00Z'}
   ⚠️ No data for sensor 35

📡 Fetching sensor 12234787 (hourly)...
   URL: https://api.openaq.org/v3/sensors/12234787/measurements
   Params: {'limit': 100, 'page': 1, 'interval': 'hourly', 'datetime_from': '2023-01-01T00:00:00Z', 'dateti

In [9]:
!ls -lt data/raw | head -n 5

total 1133168
-rw-rw-r-- 1 parallels parallels 13541796 Jan 23 06:22 sensor_5077838_hourly_20260123_062240.json
-rw-rw-r-- 1 parallels parallels   811760 Jan 23 06:15 sensor_5079217_hourly_20260123_061512.json
-rw-rw-r-- 1 parallels parallels  3424628 Jan 23 06:15 sensor_220_hourly_20260123_061504.json
-rw-rw-r-- 1 parallels parallels  1204407 Jan 23 06:14 sensor_219_hourly_20260123_061417.json
ls: write error: Broken pipe


## 2) ETL (Raw JSON -> Parquet)
Clean and normalize the raw data, then write Parquet.


In [ ]:
!python src/etl.py

In [20]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-etl').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_clean = spark.read.parquet('data/processed/pm25_clean')
print('Clean records:', df_clean.count())
df_clean.show(5, truncate=False)
spark.stop()

Clean records: 997568
+---------+-------------------+-------+
|sensor_id|datetime           |pm25   |
+---------+-------------------+-------+
|61       |2022-12-31 17:00:00|52.1042|
|61       |2022-12-31 18:00:00|36.091 |
|61       |2022-12-31 19:00:00|44.4308|
|61       |2022-12-31 20:00:00|17.422 |
|61       |2022-12-31 22:00:00|0.41998|
+---------+-------------------+-------+
only showing top 5 rows



## 3) EDA (Optional)
Run Spark SQL summaries (daily/weekly trends).


In [11]:
!python src/eda.py

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/22 17:47:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/22 17:47:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/22 17:47:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/22 17:47:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/22 17:47:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/22 17:47:07 WARN WindowExec: No Partition Defined for Window 

## 4) Feature Engineering
Create time features and lag_1..lag_24 for time-series learning.


In [12]:
!python src/features.py

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/22 17:48:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
📂 Reading hourly data...
   Hourly records: 997568

🔧 Creating features...
   Created lag_6 features...
   Created lag_12 features...
   Created lag_18 features...
   Created lag_24 features...

🧹 Dropping rows with null lag values...
26/01/22 17:48:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
   Features records after dropna: 995408                                        

💾 Saving features...
26/01/22 17:49:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                             

In [13]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-features').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_feat = spark.read.parquet('data/processed/features')
print('Feature records:', df_feat.count())
print('Columns:', df_feat.columns)
df_feat.show(3, truncate=False)
spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/22 17:53:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Feature records: 995408
Columns: ['sensor_id', 'datetime', 'pm25', 'hour', 'day_of_week', 'month', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24']


26/01/22 17:53:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+-------------------+-------+----+-----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|sensor_id|datetime           |pm25   |hour|day_of_week|month|lag_1|lag_2|lag_3|lag_4|lag_5|lag_6|lag_7|lag_8|lag_9|lag_10|lag_11|lag_12|lag_13|lag_14|lag_15|lag_16|lag_17|lag_18|lag_19|lag_20|lag_21|lag_22|lag_23|lag_24|
+---------+-------------------+-------+----+-----------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|4154     |2024-06-10 09:00:00|0.184  |9   |2          |6    |0.058|0.058|1.274|1.274|4.025|4.025|2.046|2.046|2.206|2.206 |3.373 |3.373 |2.104 |2.104 |4.263 |4.263 |1.449 |1.449 |3.171 |3.171 |1.103 |1.103 |2.094 |2.094 |
|4154     |2024-06-10 09:00:00|0.184  |9   |2          |6    |0.184|0.058|0.058|1.274|1.274|4.025|4.025|2.046|2.

## 5) Train Model (Spark MLlib)
Time-based split to avoid leakage.


In [14]:
!python src/train.py

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/22 17:54:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
📂 Loading feature data...
   Total records: 995408

🔧 Creating feature vector with 27 features:
   ['hour', 'day_of_week', 'month', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10', 'lag_11', 'lag_12', 'lag_13', 'lag_14', 'lag_15', 'lag_16', 'lag_17', 'lag_18', 'lag_19', 'lag_20', 'lag_21', 'lag_22', 'lag_23', 'lag_24']

⏰ Time-based split (no data leakage)...
26/01/22 17:54:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
   Total valid records: 995408                                                  
                                                     

In [16]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-metrics').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_metrics = spark.read.parquet('data/processed/model_metrics')
df_metrics.show(truncate=False)
spark.stop()

+-----------------+------------------+------------------+------------------+
|model            |rmse              |mae               |r2                |
+-----------------+------------------+------------------+------------------+
|linear_regression|3.4353529678717494|1.1873344366697842|0.9604698241719575|
+-----------------+------------------+------------------+------------------+



## 6) Predict
Load model and write predictions.


In [17]:
!python src/predict.py

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/22 17:57:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Feature data loaded
26/01/22 17:57:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
+---------+-------------------+-------+----+-----------+-----+-------+-------+-----+-----+-----+-----+-----+-----+-----+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+
|sensor_id|           datetime|   pm25|hour|day_of_week|month|  lag_1|  lag_2|lag_3|lag_4|lag_5|lag_6|lag_7|lag_8|lag_9|lag_10|lag_11|lag_12|lag_13|lag_14|lag_15|lag_16|lag_17|lag_18|lag_19|lag_20|lag_21|lag_22|lag_23|lag_24|
+---------+-------------------+-------+----+-----------+-----+----

In [18]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('demo-predict').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('WARN')
df_pred = spark.read.parquet('data/processed/predictions')
df_pred.show(10, truncate=False)
spark.stop()

+-------------------+-----------+------------------+
|date               |actual_pm25|predicted_pm25    |
+-------------------+-----------+------------------+
|2024-06-10 09:00:00|0.184      |0.5605604935365068|
|2024-06-10 09:00:00|0.184      |0.5743053627735143|
|2024-06-10 12:00:00|2.30158    |0.6084420127363109|
|2024-06-10 12:00:00|2.30158    |2.3490411271288782|
|2024-06-10 14:00:00|2.302      |2.4223585819773317|
|2024-06-10 14:00:00|2.302      |2.5304245095756026|
|2024-06-10 15:00:00|2.626      |2.531331517210526 |
|2024-06-10 15:00:00|2.626      |2.8424802344973528|
|2024-06-10 16:00:00|2.866      |2.8354401177802093|
|2024-06-10 16:00:00|2.866      |3.043298376855897 |
+-------------------+-----------+------------------+
only showing top 10 rows



## Demo Wrap-up
Key message: The strength of this project is the Big Data pipeline and time-series feature engineering.
